# 04c -- dim_dynasty_metric (metric_key index)

**Purpose:** Curated dimension for the `metric_key` field of
`fact_dynasty_ranking_metrics`. Lets Power BI use metrics as a **matrix column
axis** (`metric_label`) with **`metric_order`** controlling left-to-right flow
(set `metric_label`'s *Sort by column* = `metric_order`). `metric_group` gives an
optional column-group level; `direction` drives conditional formatting.

This is a hand-maintained transformer/seed (like `dim_position`): edit the SEED to
relabel, regroup, or reorder. The validation cell flags any metric_key present in
the fact but missing here (add a row) or defined here but unused.

**Output:** `data/dim_dynasty_metric.parquet` (PK `metric_key`).
Run with CWD = repo root.

In [1]:
# ---- Setup & Config ---------------------------------------------------------
from pathlib import Path
import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG

TABLE = "dim_dynasty_metric"
print("building", TABLE)

building dim_dynasty_metric


In [2]:
# ---- Curated seed -----------------------------------------------------------
# (metric_key, metric_label, metric_group, metric_order, value_type, direction, source_name)
# direction: "up" higher=better | "down" lower=better | "neutral".
# source_name: the ranking source that owns the metric (one source per key). "Composite"
#   = derived cross-source blend (written by 04y). metric_order grouped in 10s with gaps.

# The 6 per-source Rank rows are GENERATED from etl.SOURCE_PREFIX (not hand-typed) so
# this seed and 04b/04x/fold_ranks_long share one source list -- adding a ranking source
# is a single edit to SOURCE_PREFIX and its overall/positional rows appear here for free.
# The prefix IS the lowercase display abbrev (ktc/ds/fp), so .upper() gives the label
# token (KTC/DS/FP) without a second map. Published ranks are lower = better -> "down".
# metric_order: all overall ranks (1..N by source order), then all positional (N+1..2N).
RANK_KINDS = [("overall_rank", "Overall Rank"), ("positional_rank", "Pos. Rank")]
RANK_ROWS = []
for _kind_idx, (_key, _disp) in enumerate(RANK_KINDS):
    for _src_idx, (_source, _prefix) in enumerate(etl.SOURCE_PREFIX.items()):
        _order = _kind_idx * len(etl.SOURCE_PREFIX) + _src_idx + 1
        RANK_ROWS.append(
            (f"{_prefix}_{_key}", f"{_prefix.upper()} {_disp}", "Rank", _order, "num", "down", _source))

SEED = RANK_ROWS + [
    # Value
    ("value",                   "KTC Value",            "Value",      10, "num",  "up",      "KTC"),
    ("ds_value",                "DS 3D Value+",         "Value",      11, "num",  "up",      "DynastySharks"),
    # Market -- ADP is source-specific (crowd vs projection); blended into composite_adp
    ("ktc_adp",                 "KTC ADP",              "Market",     20, "num",  "down",    "KTC"),
    ("ds_adp",                  "DS ADP",               "Market",     21, "num",  "down",    "DynastySharks"),
    ("composite_adp",           "Composite ADP",        "Market",     22, "num",  "down",    "Composite"),
    ("sources_count",           "ADP Sources",          "Market",     23, "num",  "up",      "Composite"),
    ("startup_adp",             "Startup ADP",          "Market",     24, "num",  "down",    "KTC"),
    ("avg_auction_pct",         "Auction %",            "Market",     25, "num",  "up",      "KTC"),
    ("startup_avg_auction_pct", "Startup Auction %",    "Market",     26, "num",  "up",      "KTC"),
    ("std_liquidity",           "Liquidity",            "Market",     27, "num",  "neutral", "KTC"),
    # Tier (1 = best)
    ("overall_tier",            "Overall Tier",         "Tier",       30, "num",  "down",    "KTC"),
    ("positional_tier",         "Positional Tier",      "Tier",       31, "num",  "down",    "KTC"),
    # Projections (DynastySharks, fantasy points)
    ("proj_1yr",                "Proj 1-Yr",            "Projection", 40, "num",  "up",      "DynastySharks"),
    ("proj_3yr",                "Proj 3-Yr",            "Projection", 41, "num",  "up",      "DynastySharks"),
    ("proj_5yr",                "Proj 5-Yr",            "Projection", 42, "num",  "up",      "DynastySharks"),
    ("proj_10yr",               "Proj 10-Yr",           "Projection", 43, "num",  "up",      "DynastySharks"),
    # Consensus distribution (FantasyPros, rank units -> lower better)
    ("avg",                     "FP Avg Rank",          "Consensus",  50, "num",  "down",    "FantasyPros"),
    ("best",                    "FP Best Rank",         "Consensus",  51, "num",  "down",    "FantasyPros"),
    ("worst",                   "FP Worst Rank",        "Consensus",  52, "num",  "down",    "FantasyPros"),
    ("stddev",                  "FP Std Dev",           "Consensus",  53, "num",  "down",    "FantasyPros"),
    # Trends (KTC)
    ("overall_trend",           "Overall Trend",        "Trend",      60, "num",  "up",      "KTC"),
    ("positional_trend",        "Pos. Trend",           "Trend",      61, "num",  "up",      "KTC"),
    ("overall_7day_trend",      "Overall Trend (7d)",   "Trend",      62, "num",  "up",      "KTC"),
    ("positional_7day_trend",   "Pos. Trend (7d)",      "Trend",      63, "num",  "up",      "KTC"),
    # Crowd (KTC rating game)
    ("kept",                    "Kept",                 "Crowd",      70, "num",  "up",      "KTC"),
    ("traded",                  "Traded",               "Crowd",      71, "num",  "neutral", "KTC"),
    ("cut",                     "Cut",                  "Crowd",      72, "num",  "down",    "KTC"),
    # Notes
    ("analysis",                "Analysis",             "Notes",      90, "text", "neutral", "DynastySharks"),
]

dim = pd.DataFrame(SEED, columns=[
    "metric_key", "metric_label", "metric_group", "metric_order", "value_type",
    "direction", "source_name"])
assert dim["metric_key"].is_unique, "metric_key must be unique (PK)"
assert dim["metric_order"].is_unique, "metric_order should be unique for a clean sort"
print(f"{len(dim)} metric definitions ({len(RANK_ROWS)} generated rank rows + {len(SEED) - len(RANK_ROWS)} bespoke)")

34 metric definitions (6 generated rank rows + 28 bespoke)


In [3]:
# ---- Validate against the fact, then write ----------------------------------
fact_path = f"{CFG.data_dir}/fact_dynasty_ranking_metrics.parquet"
if Path(fact_path).exists():
    fact_keys = set(pd.read_parquet(fact_path, columns=["metric_key"])["metric_key"].unique())
    seed_keys = set(dim["metric_key"])
    missing = sorted(fact_keys - seed_keys)   # in fact, not defined -> ADD a row
    unused  = sorted(seed_keys - fact_keys)   # defined, not (yet) in fact -> ok
    if missing:
        print(f"[WARN] {len(missing)} metric_key in fact but missing from dim (add rows): {missing}")
    else:
        print(f"[ok] all {len(fact_keys)} fact metric_keys are defined in the dim")
    if unused:
        print(f"[info] {len(unused)} defined-but-unused (future sources): {unused}")
else:
    print("[info] fact_dynasty_ranking_metrics not found — skipping coverage check")

out = f"{CFG.data_dir}/{TABLE}.parquet"
dim.to_parquet(out, index=False)
print(f"[ok] wrote {len(dim)} rows -> {out}")

[ok] all 34 fact metric_keys are defined in the dim
[ok] wrote 34 rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data/dim_dynasty_metric.parquet


In [4]:
# ---- Preview (matrix column flow = metric_order) ----------------------------
print(dim.sort_values("metric_order").to_string(index=False))

             metric_key       metric_label metric_group  metric_order value_type direction   source_name
       ktc_overall_rank   KTC Overall Rank         Rank             1        num      down           KTC
        ds_overall_rank    DS Overall Rank         Rank             2        num      down DynastySharks
        fp_overall_rank    FP Overall Rank         Rank             3        num      down   FantasyPros
    ktc_positional_rank      KTC Pos. Rank         Rank             4        num      down           KTC
     ds_positional_rank       DS Pos. Rank         Rank             5        num      down DynastySharks
     fp_positional_rank       FP Pos. Rank         Rank             6        num      down   FantasyPros
                  value          KTC Value        Value            10        num        up           KTC
               ds_value       DS 3D Value+        Value            11        num        up DynastySharks
                ktc_adp            KTC ADP       Market